In [ ]:
from langgraph.graph import StateGraph , START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI(model = 'gpt-4o-mini')

In [ ]:
class EvaluationSchema(BaseModel):
    
    feedback: str = Field(description='Detailed feedback for essay')
    score : int = Field(description='Score out of 10', ge =10 ,le = 0)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = "blah blah essay on the some topic"

In [ ]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign score out of 10 (in integer)\n{essay}'
structured_model.invoke(prompt).score

In [ ]:
class UPSCState(TypedDict):
    essay : str
    language_feedback : str
    depth_of_analysis_feedback : str
    clarity_of_analysis_feedback : str
    overall_feedback : str
    individual_score : Annotated[list[int], operator.add] # here we ,make a 1 integer list from all three inputs and then merge(add) them all into a single list
    avg_score : float

In [ ]:
def evaluate_language(state : UPSCState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign score out of 10 (in integer)\n{state['language_feedback']}'
    output = structured_model.invoke(prompt)
    
    return {'language_feedback': output.feedback , 'individual_score':[output.score]}

def evaluate_depth(state : UPSCState):
    prompt = f'Evaluate the Analysis of Depth of the following essay and provide a feedback and assign score out of 10 (in integer)\n{state['depth_of_analysis_feedback']}'
    output = structured_model.invoke(prompt)
    
    return {'depth_of_analysis_feedback': output.feedback , 'individual_score':[output.score]}

def evaluate_clarity(state : UPSCState):
    prompt = f'Evaluate the Clarity of the following essay and provide a feedback and assign score out of 10 (in integer)\n{state['clarity_of_analysis_feedback']}'
    output = structured_model.invoke(prompt)
    
    return {'clarity_of_analysis_feedback': output.feedback , 'individual_score':[output.score]}

def final_evaluation(state : UPSCState):
    # summary
    prompt = f'Give the overall summary of the following feedbacks \n language feedback = {state['language_feedback']} \n Depth of analysis feedback = {state['depth_of_analysis_feedback']} \n Depth of clarity feedback = {state['clarity_of_analysis_feedback']}
    overall_feedback = model.invoke(prompt).content
    
    # avg calculate
    avg_score = sum(state['individual_score'])/len(state['individual_score'])
    
    
    return {'overall_feedback': overall_feedback , 'avg_Score':avg_score}



In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_depth', evaluate_depth)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

#graph edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_depth')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_depth', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {
    'essay' : essay
}

workflow.invoke(initial_state)